# Inspect One Masked Event Batch - v1

This notebook loads one event-chunk batch, checks tensor shapes and finite values, optionally loads a checkpoint, renders Keras-style model summary/graph artifacts, runs one masked-autoencoder inference, and computes one reconstruction loss.


In [ ]:
from pathlib import Path

MODEL_VERSION = "v1"
WORKSPACE = Path(r"D:\TradingCodes\quant-research-workbench")
MODEL_ROOT = Path(r"D:\TradingData\quant-research-workbench\market_data\models\masked_event_model") / MODEL_VERSION
CACHE_ROOT = Path(r"D:\market-data\flatfiles\us_stocks_sip\derived\event_chunks_v2")
CANONICAL_ROOT = Path(r"D:\market-data\flatfiles\us_stocks_sip\derived\canonical_events_v2")

# Edit these values for the batch/checkpoint you want to inspect.
TRAIN_START_DATE = "2025-11-01"
TRAIN_END_DATE = "2025-11-30"
VALIDATION_START_DATE = "2025-12-01"
VALIDATION_END_DATE = "2025-12-05"
TEST_START_DATE = "2025-12-08"
TEST_END_DATE = "2025-12-12"
TICKERS = ("ALL",)
BATCH_SIZE = 8
CONTEXT_SECONDS = 30
CHUNK_MS = 500
DEVICE = "cuda"
USE_AMP = True

# Set to None to inspect a randomly initialized model.
CHECKPOINT_PATH = None
# CHECKPOINT_PATH = MODEL_ROOT / "mem-v1-d512-e2-t8-d4-mask70-chunk500-nov2025" / "checkpoint_last.pt"

ARCHITECTURE_OUTPUT_DIR = MODEL_ROOT / "inspect_architecture"


In [ ]:
import sys
import json
import numpy as np
import polars as pl
import torch
from torch.utils.data import DataLoader

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from research.masked_event_model.v1.config import DataConfig, ModelConfig, MaskConfig, LossConfig
from research.masked_event_model.v1.data import EventChunkDataset, discover_chunk_files, target_horizons_from_columns
from research.masked_event_model.v1.masking import build_structured_masks
from research.masked_event_model.v1.model import MaskedEventAutoencoder
from research.masked_event_model.v1.model_artifacts import save_model_architecture_artifacts
from research.masked_event_model.v1.losses import masked_autoencoder_loss
from research.masked_event_model.v1.schema import QUOTE_FEATURE_COLUMNS, TRADE_FEATURE_COLUMNS, CHUNK_SUMMARY_COLUMNS

np.set_printoptions(precision=5, suppress=True)
torch.set_float32_matmul_precision("high")
device = torch.device(DEVICE if DEVICE == "cpu" or torch.cuda.is_available() else "cpu")
print("device:", device)


data_config = DataConfig(
    cache_root=CACHE_ROOT,
    canonical_root=CANONICAL_ROOT,
    train_start_date=TRAIN_START_DATE,
    train_end_date=TRAIN_END_DATE,
    validation_start_date=VALIDATION_START_DATE,
    validation_end_date=VALIDATION_END_DATE,
    test_start_date=TEST_START_DATE,
    test_end_date=TEST_END_DATE,
    tickers=TICKERS,
    context_seconds=CONTEXT_SECONDS,
    chunk_ms=CHUNK_MS,
    shuffle_files=False,
    shuffle_windows=False,
)
model_config = ModelConfig()
mask_config = MaskConfig()
loss_config = LossConfig()

files = discover_chunk_files(data_config, start_date=TRAIN_START_DATE, end_date=TRAIN_END_DATE)
print("chunk files:", len(files))
if not files:
    raise FileNotFoundError(f"No chunk files found under {data_config.cache_root}")

schema = pl.scan_parquet(str(files[0].path)).collect_schema()
horizons = target_horizons_from_columns(schema.names())
horizon_count = len(horizons)
print("first file:", files[0].path)
print("target horizons:", horizons)
print("context_chunks:", data_config.context_chunks)
print("quote features:", len(QUOTE_FEATURE_COLUMNS), "trade features:", len(TRADE_FEATURE_COLUMNS), "summary features:", len(CHUNK_SUMMARY_COLUMNS))


In [ ]:
dataset = EventChunkDataset(
    config=data_config,
    split="train",
    batch_size=BATCH_SIZE,
    seed=17,
)
loader = DataLoader(dataset, batch_size=None, num_workers=0)
batch = next(iter(loader))
print("batch keys:", sorted(batch.keys()))
for key, value in batch.items():
    if torch.is_tensor(value):
        finite = bool(torch.isfinite(value).all()) if value.is_floating_point() else True
        if value.numel() and value.is_floating_point():
            print(f"{key:24s} shape={tuple(value.shape)} dtype={value.dtype} finite={finite} min={float(value.min()):.6g} max={float(value.max()):.6g}")
        else:
            print(f"{key:24s} shape={tuple(value.shape)} dtype={value.dtype} finite={finite}")
    else:
        print(f"{key:24s} type={type(value).__name__}")


In [ ]:
model = MaskedEventAutoencoder(
    quote_feature_count=len(QUOTE_FEATURE_COLUMNS),
    trade_feature_count=len(TRADE_FEATURE_COLUMNS),
    summary_feature_count=len(CHUNK_SUMMARY_COLUMNS),
    context_chunks=data_config.context_chunks,
    max_quote_events=data_config.max_quote_events,
    max_trade_events=data_config.max_trade_events,
    max_total_events=data_config.max_total_events,
    horizon_count=horizon_count,
    target_bit_count=data_config.target_bit_count,
    config=model_config,
).to(device)

checkpoint = None
if CHECKPOINT_PATH is not None:
    checkpoint_path = Path(CHECKPOINT_PATH)
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    state_dict = checkpoint.get("model", checkpoint.get("model_state_dict", checkpoint))
    model.load_state_dict(state_dict, strict=True)
    print(f"loaded checkpoint: {checkpoint_path}")
    if isinstance(checkpoint, dict) and "step" in checkpoint:
        print("checkpoint step:", checkpoint["step"])
else:
    print("using randomly initialized model")

param_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"parameters={param_count:,} trainable={trainable_count:,}")


In [ ]:
# Keras-style model summary and graph view.
# Optional packages for richer output:
#   pip install torchinfo torchview graphviz
# Graph rendering also needs the Graphviz dot executable on PATH.

from IPython.display import Markdown, Image, display

ARCHITECTURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
architecture_info = save_model_architecture_artifacts(
    model=model,
    data_config=data_config,
    output_dir=ARCHITECTURE_OUTPUT_DIR,
    version="mem-v1-inspect",
    torch_module=torch,
    wandb_run=None,
    summary_batch_size=1,
    summary_depth=8,
    graph_depth=3,
)

summary_path = Path(architecture_info["summary_path"])
metadata_path = ARCHITECTURE_OUTPUT_DIR / "model_architecture" / "model_architecture.json"
print("summary:", summary_path)
print("metadata:", metadata_path)
print("graph png:", architecture_info.get("graph_png_path"))
print("errors:", architecture_info.get("errors"))

if summary_path.exists():
    text = summary_path.read_text(encoding="utf-8")
    display(Markdown("```text\n" + text[:20000] + ("\n... truncated ..." if len(text) > 20000 else "") + "\n```"))

png_path = architecture_info.get("graph_png_path")
if png_path and Path(png_path).exists():
    display(Image(filename=str(png_path)))


In [ ]:
def move_tensor_batch(batch_dict, device):
    return {key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value for key, value in batch_dict.items()}

model.eval()
device_batch = move_tensor_batch(batch, device)
masks = build_structured_masks(
    quote_values=device_batch["quote_values"],
    trade_values=device_batch["trade_values"],
    chunk_summary=device_batch["chunk_summary"],
    event_kinds=device_batch["event_kinds"],
    config=mask_config,
)
with torch.inference_mode():
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP and device.type == "cuda"):
        output = model(
            device_batch["quote_values"],
            device_batch["trade_values"],
            device_batch["event_kinds"],
            device_batch["event_indices"],
            device_batch["chunk_summary"],
            masks,
        )
        loss, metrics = masked_autoencoder_loss(output, device_batch, masks, loss_config)

print("loss:", float(loss.detach().cpu()))
print(json.dumps(metrics, indent=2, sort_keys=True))
print("embedding shape:", tuple(output.embeddings.shape))
print("encoded chunks shape:", tuple(output.encoded_chunks.shape))
print("forecast logits shape:", tuple(output.forecast_logits.shape))


In [ ]:
# Quick target/prediction bit sanity check for the first sample.
print("target bits first sample horizon 1:", batch["targets"][0, 0, 0].numpy())
print("forecast logits first sample horizon 1:", output.forecast_logits.detach().cpu()[0, 0, 0].numpy())
print("target_bps first sample:", batch["target_bps"][0].numpy())
